# Robust Weak-Preference RSD: Pareto Efficiency and Stability


This notebook is intentionally RSD-only. It trains a coherent ordinal bundle-score proxy from A/B/INDIFFERENT pair labels, calibrates separate thresholds for Pareto efficiency and stability, and evaluates repeated held-out markets. It never uses cardinal allocation utility to select the RSD model.


## 1. Install packages


In [ ]:
!python -m pip install -q -U "transformers>=4.56,<5" "accelerate>=1.2" "bitsandbytes>=0.46.1" scikit-learn scipy sentencepiece gdown

In [ ]:
import ast
import json
import random
import re
import shutil
import time
from itertools import combinations
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import lil_matrix
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm

SEED = 20260909
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
N_FOLDS = 5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## 2. Load the synthetic course-allocation data


In [ ]:
from pathlib import Path
import json
import os
import shutil
import zipfile

import gdown
import numpy as np
import pandas as pd

# Use a local dataset ZIP, or opt in to a link-shared Drive download.
ZIP_PATH = Path("fixed_tolerance_course_data.zip")
FILE_ID = os.environ.get("COURSE_DATA_DRIVE_FILE_ID", "").strip()

if not ZIP_PATH.exists() or not zipfile.is_zipfile(ZIP_PATH):
    if not FILE_ID:
        raise FileNotFoundError(
            f"Place a valid {ZIP_PATH} in the working directory or set "
            "COURSE_DATA_DRIVE_FILE_ID to the ID of a link-shared dataset."
        )

    downloaded_path = gdown.download(
        id=FILE_ID,
        output=str(ZIP_PATH),
        quiet=False,
    )
    if downloaded_path is None:
        raise RuntimeError(
            "The Google Drive download failed. Check that anyone with the link "
            "can download the configured file."
        )

assert ZIP_PATH.exists(), f"Downloaded file not found: {ZIP_PATH}"
assert zipfile.is_zipfile(ZIP_PATH), (
    "The downloaded file is not a valid ZIP. Check the Google Drive "
    "sharing permission and FILE_ID."
)

print(
    f"Dataset ZIP: {ZIP_PATH} "
    f"({ZIP_PATH.stat().st_size / 1_000_000:.2f} MB)"
)

# Extract the dataset into a clean input directory.
INPUT_DIR = Path("fixed_tolerance_input")

if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)

INPUT_DIR.mkdir(parents=True)
shutil.unpack_archive(str(ZIP_PATH), INPUT_DIR)

# Locate the required files even if the ZIP contains a parent folder.
found = {
    path.name: path
    for path in INPUT_DIR.rglob("*")
    if path.is_file()
}

required = {
    "course_catalog.csv",
    "student_reports.csv",
    "student_ground_truth.json",
    "bundle_queries_with_truth.csv",
    "generation_metadata.json",
}

missing = required - set(found)

assert not missing, (
    f"Missing required files: {sorted(missing)}. "
    f"Files found: {sorted(found)}"
)

# Load the dataset.
catalog = pd.read_csv(found["course_catalog.csv"])
reports_df = pd.read_csv(found["student_reports.csv"])
queries = pd.read_csv(found["bundle_queries_with_truth.csv"])

metadata = json.loads(
    found["generation_metadata.json"].read_text(encoding="utf-8")
)

ground_truth = json.loads(
    found["student_ground_truth.json"].read_text(encoding="utf-8")
)

reports = (
    reports_df
    .set_index("student_id")["language_report"]
    .to_dict()
)

COURSE_IDS = sorted(
    catalog["course_id"].astype(int).tolist()
)

MAX_BUNDLE_SIZE = int(
    metadata["maximum_bundle_size"]
)

TRUE_COMMON_TOLERANCE = float(
    metadata["common_tolerance"]
)

# Convert serialized bundle lists back into Python lists.
for column in ["bundle_a", "bundle_b"]:
    queries[column] = queries[column].apply(json.loads)

# Student splits are generated later inside the mechanism-specific experiment.
student_ids = np.array(sorted(reports_df["student_id"].astype(int).unique()))
print("Students:", len(student_ids))
print("Queries:", len(queries))
print("True tolerance (used only to recover ordinal ground-truth tiers):", TRUE_COMMON_TOLERANCE)

display(catalog)
display(reports_df.sample(3, random_state=SEED))
display(queries.groupby("true_label").size().rename("count"))


## 3. Load the open-source LLM

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), "Select a GPU runtime in Colab."

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
).eval()

MODEL_REVISION = getattr(model.config, "_commit_hash", None)
print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("GPU:", torch.cuda.get_device_name(0))

## 4. Structured profile prompt and robust validation

In [ ]:
PUBLIC_CATALOG = catalog[
    ["course_id", "course_name", "field", "time_slot", "workload", "description"]
].to_dict("records")

PROFILE_PROMPT = '''
You extract a structured course-preference profile from a student's report.
Use only the report and the public course catalog. Never invent preferences.

For every course with clear evidence, assign exactly one attitude from:
STRONGLY_DISLIKE, DISLIKE, NEUTRAL, LIKE, STRONGLY_LIKE.
Omit courses without clear evidence; omitted courses are treated as NEUTRAL.

Extract complements only when two courses are especially valuable together.
Extract substitutes only when the student indicates overlap, redundancy, or a
desire not to take both. Represent a pair as "smaller_id|larger_id".

Return exactly one JSON object:
{
  "course_attitudes": [{"course_id": 3, "attitude": "LIKE"}],
  "complements": ["0|6"],
  "substitutes": ["2|7"]
}
Do not return Markdown, explanations, confidence scores, tolerance estimates,
or additional keys.
'''.strip()

ATTITUDE_VALUES = {
    "STRONGLY_DISLIKE": -2.0,
    "DISLIKE": -1.0,
    "NEUTRAL": 0.0,
    "LIKE": 1.0,
    "STRONGLY_LIKE": 2.0,
}
ATTITUDE_ALIASES = {
    "STRONG_DISLIKE": "STRONGLY_DISLIKE",
    "VERY_DISLIKE": "STRONGLY_DISLIKE",
    "STRONG_LIKE": "STRONGLY_LIKE",
    "VERY_LIKE": "STRONGLY_LIKE",
}


def extract_json_object(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)
    start = text.find("{")
    end = text.rfind("}")
    if start < 0 or end < start:
        raise ValueError("No JSON object found.")
    return json.loads(text[start:end + 1])


def canonical_pair(value):
    if isinstance(value, str):
        pieces = value.split("|")
    elif isinstance(value, (list, tuple)) and len(value) == 2:
        pieces = value
    elif isinstance(value, dict):
        pieces = [value.get("course_id_1"), value.get("course_id_2")]
    else:
        raise ValueError(f"Invalid pair: {value}")
    a, b = sorted(map(int, pieces))
    if a == b or a not in COURSE_IDS or b not in COURSE_IDS:
        raise ValueError(f"Invalid course pair: {value}")
    return (a, b)


def validate_profile(parsed):
    if not isinstance(parsed, dict):
        raise ValueError("The parsed response is not an object.")

    course_scores = {course_id: 0.0 for course_id in COURSE_IDS}
    explicitly_scored = set()
    for item in parsed.get("course_attitudes", []):
        course_id = int(item["course_id"])
        attitude = str(item["attitude"]).strip().upper().replace(" ", "_")
        attitude = ATTITUDE_ALIASES.get(attitude, attitude)
        if course_id not in COURSE_IDS:
            raise ValueError(f"Unknown course ID: {course_id}")
        if attitude not in ATTITUDE_VALUES:
            raise ValueError(f"Unknown attitude: {attitude}")
        course_scores[course_id] = ATTITUDE_VALUES[attitude]
        explicitly_scored.add(course_id)

    complements = sorted(set(canonical_pair(x) for x in parsed.get("complements", [])))
    substitutes = sorted(set(canonical_pair(x) for x in parsed.get("substitutes", [])))
    overlap = set(complements) & set(substitutes)
    if overlap:
        # Ambiguous relationships are conservatively discarded.
        complements = [pair for pair in complements if pair not in overlap]
        substitutes = [pair for pair in substitutes if pair not in overlap]

    return {
        "course_scores": course_scores,
        "explicitly_scored_courses": sorted(explicitly_scored),
        "complements": complements,
        "substitutes": substitutes,
    }


@torch.inference_mode()
def generate_profile(report, max_new_tokens=420):
    payload = {"course_catalog": PUBLIC_CATALOG, "student_report": report}
    messages = [
        {"role": "system", "content": PROFILE_PROMPT},
        {"role": "user", "content": json.dumps(payload, ensure_ascii=False)},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated = output[0, inputs.input_ids.shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
    parsed = extract_json_object(raw)
    return validate_profile(parsed), raw

## 5. Smoke test

In [ ]:
smoke_student_id = int(reports_df.student_id.iloc[0])
smoke_profile, smoke_raw = generate_profile(reports[smoke_student_id])
print("Raw output:")
print(smoke_raw)
print("\nValidated profile:")
print(smoke_profile)

## 6. Extract one profile per student with checkpointing

In [ ]:
OUTPUT_DIR = Path("fixed_tolerance_experiment_results")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT = OUTPUT_DIR / "profile_extractions.jsonl"

completed = {}
if CHECKPOINT.exists():
    for line in CHECKPOINT.read_text(encoding="utf-8").splitlines():
        if line.strip():
            item = json.loads(line)
            completed[int(item["student_id"])] = item
print("Loaded checkpoint rows:", len(completed))

for student_id in tqdm(sorted(map(int, reports)), desc="Extracting profiles"):
    if student_id in completed and completed[student_id].get("status") == "ok":
        continue
    started = time.perf_counter()
    try:
        profile, raw = generate_profile(reports[student_id])
        item = {
            "student_id": student_id,
            "status": "ok",
            "profile": profile,
            "raw_output": raw,
            "error": None,
            "seconds": time.perf_counter() - started,
        }
    except Exception as exc:
        item = {
            "student_id": student_id,
            "status": "error",
            "profile": None,
            "raw_output": None,
            "error": repr(exc),
            "seconds": time.perf_counter() - started,
        }
    completed[student_id] = item
    with CHECKPOINT.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(item) + "\n")

status_df = pd.DataFrame([
    {key: value for key, value in item.items() if key not in {"profile", "raw_output"}}
    for item in completed.values()
]).sort_values("student_id")
display(status_df)
print(status_df.status.value_counts())

failed = status_df.loc[status_df.status != "ok", "student_id"].tolist()
assert not failed, f"Profile extraction failed for students: {failed}. Re-run this cell to retry them."

## 7. Convert profiles and bundle pairs into model features

In [ ]:
profiles = {student_id: item["profile"] for student_id, item in completed.items()}


def bundle_features(profile, bundle):
    bundle = tuple(sorted(map(int, bundle)))
    pairs = set(combinations(bundle, 2))
    score_sum = sum(float(profile["course_scores"].get(str(j), profile["course_scores"].get(j, 0.0))) for j in bundle)
    complement_count = sum(tuple(pair) in pairs for pair in profile["complements"])
    substitute_count = sum(tuple(pair) in pairs for pair in profile["substitutes"])
    explicit_count = sum(j in set(profile["explicitly_scored_courses"]) for j in bundle)
    return np.array([
        score_sum / MAX_BUNDLE_SIZE,
        complement_count / MAX_BUNDLE_SIZE,
        substitute_count / MAX_BUNDLE_SIZE,
        explicit_count / MAX_BUNDLE_SIZE,
    ], dtype=float)


feature_rows = []
for row in queries.itertuples(index=False):
    profile = profiles[int(row.student_id)]
    feature_difference = bundle_features(profile, row.bundle_a) - bundle_features(profile, row.bundle_b)
    feature_rows.append(feature_difference)

X = np.vstack(feature_rows)
FEATURE_NAMES = [
    "course_score_difference",
    "complement_difference",
    "substitute_difference",
    "explicit_coverage_difference",
]
for index, name in enumerate(FEATURE_NAMES):
    queries[name] = X[:, index]

display(queries.head())

The profile-extraction checkpoint is shared by filename across the two notebooks. If both notebooks run in the same Colab workspace, the second notebook reuses the completed LLM profiles rather than calling the GPU again.


## 8. Ordinal proxy, Pareto-improved weak RSD, and violation metrics


In [ ]:
# Experiment size. Increase repetitions after the smoke run succeeds.
N_REPEATED_SPLITS = 20
N_TRAIN_STUDENTS = 64
N_VALIDATION_STUDENTS = 32
N_TEST_STUDENTS = 32
N_VALIDATION_ORDERS = 5
N_TEST_ORDERS = 20
MARKET_CAPACITY_PER_COURSE = 5
THRESHOLD_GRID = np.round(np.arange(0.0, 0.121, 0.01), 3)
CVaR_ALPHA = 0.90
ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]

market_capacities = {int(c): MARKET_CAPACITY_PER_COURSE for c in COURSE_IDS}


def ordinal_target(labels):
    labels = np.asarray(labels)
    return np.where(labels == "A", 1.0, np.where(labels == "B", -1.0, 0.0))


def fit_ordinal_score_model(rows, return_cv=False):
    """Fit a coherent linear bundle score using only A/B/INDIFFERENT labels."""
    X = rows[FEATURE_NAMES].to_numpy()
    y = ordinal_target(rows.true_label)
    groups = rows.student_id.to_numpy()
    cv = GroupKFold(n_splits=min(N_FOLDS, rows.student_id.nunique()))
    alpha_rows = []
    for alpha in ALPHAS:
        oof = np.full(len(rows), np.nan)
        for fit_idx, hold_idx in cv.split(X, y, groups):
            model = Ridge(alpha=alpha, fit_intercept=False).fit(X[fit_idx], y[fit_idx])
            oof[hold_idx] = model.predict(X[hold_idx])
        alpha_rows.append({
            "alpha": alpha,
            "ordinal_label_mse": float(np.mean((oof - y) ** 2)),
            "strict_direction_accuracy": float(np.mean(
                np.where(oof >= 0, "A", "B") == rows.true_direction
            )),
        })
    cv_results = pd.DataFrame(alpha_rows).sort_values(
        ["ordinal_label_mse", "alpha"]
    ).reset_index(drop=True)
    best_alpha = float(cv_results.loc[0, "alpha"])
    model = Ridge(alpha=best_alpha, fit_intercept=False).fit(X, y)
    return (model, best_alpha, cv_results) if return_cv else model


def build_market_scores(rows, model):
    predicted, truth = {}, {}
    for student_id, student_rows in rows.groupby("student_id"):
        i = int(student_id)
        bundles = {
            tuple(sorted(map(int, b))) for b in student_rows.bundle_a
        } | {
            tuple(sorted(map(int, b))) for b in student_rows.bundle_b
        }
        predicted[i] = {
            b: float(np.dot(model.coef_, bundle_features(profiles[i], b)))
            for b in bundles
        }
        observations = {b: [] for b in bundles}
        for row in student_rows.itertuples(index=False):
            a = tuple(sorted(map(int, row.bundle_a)))
            b = tuple(sorted(map(int, row.bundle_b)))
            observations[a].append(float(row.true_utility_a))
            observations[b].append(float(row.true_utility_b))
        truth[i] = {b: float(np.mean(v)) for b, v in observations.items()}
        predicted[i][tuple()] = min(predicted[i].values()) - 2.0
        truth[i][tuple()] = min(truth[i].values()) - TRUE_COMMON_TOLERANCE - 1.0
    return predicted, truth


def make_tier_ranks(score_map, tolerance):
    ordered = sorted(score_map, key=lambda b: (-score_map[b], b))
    ranks, tier, anchor = {}, -1, None
    for bundle in ordered:
        score = score_map[bundle]
        if anchor is None or anchor - score > tolerance:
            tier += 1
            anchor = score
        ranks[bundle] = tier
    return ranks


def rank_maps_from_scores(score_maps, tolerance):
    return {i: make_tier_ranks(scores, tolerance) for i, scores in score_maps.items()}


def fits(bundle, remaining):
    return all(remaining[c] > 0 for c in bundle)


def base_weak_rsd(rank_maps, capacities, seed):
    rng = np.random.default_rng(seed)
    order = list(rng.permutation(sorted(rank_maps)))
    priority_position = {int(i): pos for pos, i in enumerate(order)}
    remaining = capacities.copy()
    allocation = {}
    for i in order:
        feasible = [b for b in rank_maps[i] if fits(b, remaining)]
        best_rank = min(rank_maps[i][b] for b in feasible)
        top = sorted(b for b in feasible if rank_maps[i][b] == best_rank)
        chosen = top[int(rng.integers(len(top)))]
        allocation[int(i)] = chosen
        for c in chosen:
            remaining[c] -= 1
    return allocation, priority_position


def _allocation_milp(rank_maps, capacities, allowed, objective, seed=0):
    """Solve a bundle-assignment MILP over the supplied allowed bundles."""
    students = sorted(rank_maps)
    variables = [(i, b) for i in students for b in allowed[i]]
    index = {key: k for k, key in enumerate(variables)}
    n_vars = len(variables)
    rows, lower, upper = [], [], []

    for i in students:
        row = {}
        for b in allowed[i]:
            row[index[(i, b)]] = 1.0
        rows.append(row); lower.append(1.0); upper.append(1.0)

    for course in COURSE_IDS:
        row = {}
        for i, b in variables:
            if course in b:
                row[index[(i, b)]] = 1.0
        rows.append(row); lower.append(-np.inf); upper.append(float(capacities[course]))

    matrix = lil_matrix((len(rows), n_vars), dtype=float)
    for r, entries in enumerate(rows):
        for col, value in entries.items():
            matrix[r, col] = value

    rng = np.random.default_rng(seed)
    c = np.array([objective(i, b) for i, b in variables], dtype=float)
    c += rng.uniform(0, 1e-8, size=n_vars)
    result = milp(
        c=c,
        integrality=np.ones(n_vars),
        bounds=Bounds(np.zeros(n_vars), np.ones(n_vars)),
        constraints=LinearConstraint(matrix.tocsr(), np.array(lower), np.array(upper)),
        options={"time_limit": 30.0},
    )
    if not result.success:
        return None, result.message
    chosen = {}
    for (i, b), value in zip(variables, result.x):
        if value > 0.5:
            chosen[i] = b
    return chosen, result.message


def pareto_improve_rsd(rank_maps, baseline, capacities, seed):
    """Bundle-domain extension: no student worsens relative to baseline RSD.

    Minimizing total reported tier subject to individual non-worsening returns
    a Pareto-efficient allocation on the observed bundle domain. This is the
    bundle-MILP analogue of augmenting-path/cycle elimination.
    """
    allowed = {
        i: [b for b in rank_maps[i] if rank_maps[i][b] <= rank_maps[i][baseline[i]]]
        for i in rank_maps
    }
    improved, message = _allocation_milp(
        rank_maps, capacities, allowed,
        objective=lambda i, b: float(rank_maps[i][b]), seed=seed,
    )
    return (baseline if improved is None else improved), message


def max_pareto_improvements(allocation, true_ranks, capacities):
    """Maximum number of strictly improvable students while nobody worsens."""
    allowed = {
        i: [b for b in true_ranks[i] if true_ranks[i][b] <= true_ranks[i][allocation[i]]]
        for i in true_ranks
    }
    improved, message = _allocation_milp(
        true_ranks, capacities, allowed,
        objective=lambda i, b: -float(true_ranks[i][b] < true_ranks[i][allocation[i]]),
        seed=SEED + 777,
    )
    if improved is None:
        return np.nan, message
    count = sum(true_ranks[i][improved[i]] < true_ranks[i][allocation[i]] for i in true_ranks)
    return int(count), message


def blocking_bundle_metrics(allocation, true_ranks, capacities, priority_position):
    """Count priority-feasible blocking bundles under true weak preferences."""
    holders = {c: [] for c in COURSE_IDS}
    for j, bundle in allocation.items():
        for c in bundle:
            holders[c].append(j)

    blocking = []
    for i in sorted(allocation):
        own = allocation[i]
        own_rank = true_ranks[i][own]
        for bundle, rank in true_ranks[i].items():
            if rank >= own_rank:
                continue
            can_obtain = True
            for c in bundle:
                if c in own:
                    continue
                course_holders = holders[c]
                if len(course_holders) < capacities[c]:
                    continue
                if any(priority_position[i] < priority_position[j] for j in course_holders):
                    continue
                can_obtain = False
                break
            if can_obtain:
                blocking.append((i, bundle))
    return {
        "blocking_bundles": len(blocking),
        "blocking_students": len({i for i, _ in blocking}),
        "zero_blocking": int(len(blocking) == 0),
    }


def evaluate_rsd(allocation, true_ranks, capacities, priority_position):
    pe_count, pe_status = max_pareto_improvements(allocation, true_ranks, capacities)
    metrics = {
        "pareto_improvable_students": pe_count,
        "pareto_efficient": int(pe_count == 0) if np.isfinite(pe_count) else np.nan,
        "pe_score": 1.0 - pe_count / len(allocation) if np.isfinite(pe_count) else np.nan,
        "pe_solver_status": pe_status,
    }
    metrics.update(blocking_bundle_metrics(
        allocation, true_ranks, capacities, priority_position
    ))
    return metrics


def run_pareto_improved_rsd(predicted_scores, true_scores, threshold, seed, oracle=False):
    true_ranks = rank_maps_from_scores(true_scores, TRUE_COMMON_TOLERANCE)
    decision_ranks = true_ranks if oracle else rank_maps_from_scores(predicted_scores, threshold)
    baseline, priority = base_weak_rsd(decision_ranks, market_capacities, seed)
    allocation, solver_status = pareto_improve_rsd(
        decision_ranks, baseline, market_capacities, seed + 1
    )
    metrics = evaluate_rsd(allocation, true_ranks, market_capacities, priority)
    metrics["allocation_solver_status"] = solver_status
    return allocation, metrics


def cvar(values, alpha=0.90):
    values = np.sort(np.asarray(values, dtype=float))
    start = max(0, int(np.floor(alpha * len(values))))
    return float(values[start:].mean())


## 9. Repeated train-validation splits and robust threshold selection


In [ ]:
all_ids = np.array(sorted(queries.student_id.unique()))
assert len(all_ids) == N_TRAIN_STUDENTS + N_VALIDATION_STUDENTS + N_TEST_STUDENTS
split_rng = np.random.default_rng(SEED + 500)

split_records = []
manifest_rows = []
validation_rows = []

for split_id in tqdm(range(N_REPEATED_SPLITS), desc="RSD validation splits"):
    shuffled = split_rng.permutation(all_ids)
    train_ids = set(map(int, shuffled[:N_TRAIN_STUDENTS]))
    validation_ids = set(map(int, shuffled[N_TRAIN_STUDENTS:N_TRAIN_STUDENTS + N_VALIDATION_STUDENTS]))
    test_ids = set(map(int, shuffled[-N_TEST_STUDENTS:]))
    assert not (train_ids & validation_ids or train_ids & test_ids or validation_ids & test_ids)

    for role, ids in [("train", train_ids), ("validation", validation_ids), ("test", test_ids)]:
        manifest_rows.extend({"split_id": split_id, "student_id": i, "role": role} for i in ids)

    train = queries[queries.student_id.isin(train_ids)].copy()
    validation = queries[queries.student_id.isin(validation_ids)].copy()
    model, alpha, _ = fit_ordinal_score_model(train, return_cv=True)
    predicted, truth = build_market_scores(validation, model)
    split_records.append({
        "split_id": split_id, "model": model, "alpha": alpha,
        "train_ids": train_ids, "validation_ids": validation_ids, "test_ids": test_ids,
    })

    for threshold in THRESHOLD_GRID:
        for order_run in range(N_VALIDATION_ORDERS):
            _, metrics = run_pareto_improved_rsd(
                predicted, truth, float(threshold),
                seed=SEED + 100_000 * split_id + 1_000 * order_run,
            )
            validation_rows.append({
                "split_id": split_id, "threshold": float(threshold),
                "order_run": order_run, "ridge_alpha": alpha, **metrics,
            })

split_manifest = pd.DataFrame(manifest_rows)
validation_results = pd.DataFrame(validation_rows)

threshold_summary = (
    validation_results.groupby("threshold", as_index=False)
    .agg(
        mean_pe_violations=("pareto_improvable_students", "mean"),
        mean_blocking_bundles=("blocking_bundles", "mean"),
        mean_blocking_students=("blocking_students", "mean"),
        pe_rate=("pareto_efficient", "mean"),
        zero_block_rate=("zero_blocking", "mean"),
        worst_pe_violations=("pareto_improvable_students", "max"),
        worst_blocking_bundles=("blocking_bundles", "max"),
    )
)
threshold_summary["cvar90_pe_violations"] = threshold_summary.threshold.map(
    lambda t: cvar(validation_results.loc[
        validation_results.threshold.eq(t), "pareto_improvable_students"
    ], CVaR_ALPHA)
)
threshold_summary["cvar90_blocking_bundles"] = threshold_summary.threshold.map(
    lambda t: cvar(validation_results.loc[
        validation_results.threshold.eq(t), "blocking_bundles"
    ], CVaR_ALPHA)
)

pe_choice = threshold_summary.sort_values(
    ["cvar90_pe_violations", "mean_pe_violations", "cvar90_blocking_bundles", "threshold"]
).iloc[0]
stability_choice = threshold_summary.sort_values(
    ["cvar90_blocking_bundles", "mean_blocking_bundles", "cvar90_pe_violations", "threshold"]
).iloc[0]
PE_THRESHOLD = float(pe_choice.threshold)
STABILITY_THRESHOLD = float(stability_choice.threshold)

display(threshold_summary.sort_values("threshold"))
print("PE-calibrated threshold:", PE_THRESHOLD)
print("Stability-calibrated threshold:", STABILITY_THRESHOLD)


## 10. Repeated held-out tests


In [ ]:
test_rows = []
methods = {
    "forced_strict_proxy": (0.0, False),
    "pe_calibrated_proxy": (PE_THRESHOLD, False),
    "stability_calibrated_proxy": (STABILITY_THRESHOLD, False),
    "oracle_weak_pareto_improved_rsd": (TRUE_COMMON_TOLERANCE, True),
}

for record in tqdm(split_records, desc="RSD repeated tests"):
    split_id = record["split_id"]
    test = queries[queries.student_id.isin(record["test_ids"])].copy()
    predicted, truth = build_market_scores(test, record["model"])
    for method, (threshold, oracle) in methods.items():
        for order_run in range(N_TEST_ORDERS):
            _, metrics = run_pareto_improved_rsd(
                predicted, truth, threshold,
                seed=SEED + 5_000_000 + 100_000 * split_id + order_run,
                oracle=oracle,
            )
            test_rows.append({
                "split_id": split_id, "method": method,
                "threshold": threshold, "order_run": order_run, **metrics,
            })

test_results = pd.DataFrame(test_rows)
test_summary = (
    test_results.groupby("method", as_index=False)
    .agg(
        pe_rate=("pareto_efficient", "mean"),
        mean_pe_score=("pe_score", "mean"),
        zero_block_rate=("zero_blocking", "mean"),
        median_blocking_bundles=("blocking_bundles", "median"),
        mean_blocking_bundles=("blocking_bundles", "mean"),
        p90_blocking_bundles=("blocking_bundles", lambda x: float(np.quantile(x, 0.90))),
        worst_blocking_bundles=("blocking_bundles", "max"),
        worst_blocking_students=("blocking_students", "max"),
    )
)
test_summary["cvar90_blocking_bundles"] = test_summary.method.map(
    lambda m: cvar(test_results.loc[test_results.method.eq(m), "blocking_bundles"], 0.90)
)

display(test_summary)

# Theoretical-oracle diagnostic: do not silently claim properties if the
# bundle-domain extension fails on the observed candidate-bundle domain.
oracle_rows = test_results[test_results.method.eq("oracle_weak_pareto_improved_rsd")]
print("Oracle PE violation runs:", int((oracle_rows.pareto_efficient != 1).sum()))
print("Oracle blocking violation runs:", int((oracle_rows.zero_blocking != 1).sum()))


## 11. Visualize and save RSD robustness results


In [ ]:
FIGURE_DIR = Path("rsd_robustness_figures")
FIGURE_DIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for method, group in test_results.groupby("method"):
    axes[0].hist(group.blocking_bundles, bins=20, alpha=0.35, label=method, density=True)
    values = np.sort(group.blocking_bundles.to_numpy())
    axes[1].step(values, np.arange(1, len(values) + 1) / len(values), where="post", label=method)
axes[0].set(title="Distribution of blocking bundles", xlabel="Blocking bundles", ylabel="Density")
axes[1].set(title="Empirical CDF of blocking bundles", xlabel="Blocking bundles", ylabel="Cumulative probability")
for ax in axes:
    ax.grid(alpha=0.25); ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "blocking_bundle_distribution.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
for method, group in test_results.groupby("method"):
    jitter = np.random.default_rng(SEED).normal(0, 0.05, len(group))
    ax.scatter(group.pareto_improvable_students + jitter, group.blocking_bundles,
               alpha=0.25, s=18, label=method)
ax.set(xlabel="Pareto-improvable students", ylabel="Blocking bundles",
       title="Efficiency--stability robustness trade-off")
ax.grid(alpha=0.25); ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "pe_stability_tradeoff.png", dpi=180)
plt.show()

OUTPUT_DIR = Path("rsd_weak_preference_robustness_results")
OUTPUT_DIR.mkdir(exist_ok=True)
split_manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)
validation_results.to_csv(OUTPUT_DIR / "validation_runs.csv", index=False)
threshold_summary.to_csv(OUTPUT_DIR / "threshold_summary.csv", index=False)
test_results.to_csv(OUTPUT_DIR / "test_runs.csv", index=False)
test_summary.to_csv(OUTPUT_DIR / "test_summary.csv", index=False)
for path in FIGURE_DIR.glob("*.png"):
    shutil.copy2(path, OUTPUT_DIR / path.name)

metadata = {
    "seed": SEED,
    "preference_information": "ordinal A/B/INDIFFERENT labels only",
    "mechanism": "bundle-domain Pareto-improved weak RSD",
    "bundle_extension_warning": (
        "MILP post-processing is a bundle-domain extension of augmenting-path/cycle "
        "Pareto improvement; oracle PE and stability are checked empirically."
    ),
    "n_repeated_splits": N_REPEATED_SPLITS,
    "split_sizes": {"train": N_TRAIN_STUDENTS, "validation": N_VALIDATION_STUDENTS, "test": N_TEST_STUDENTS},
    "capacity_per_course": MARKET_CAPACITY_PER_COURSE,
    "pe_threshold": PE_THRESHOLD,
    "stability_threshold": STABILITY_THRESHOLD,
    "robust_selection": f"CVaR at alpha={CVaR_ALPHA}",
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
archive = Path(shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR))
print("Saved:", archive.resolve())
from google.colab import files
files.download(str(archive))
